# UC Steward — Configuration Reference

This is the **single source of truth** for all tunable settings. Edit these values before your first run.

Settings flow: this notebook documents defaults → `databricks.yml` variables → job `base_parameters` → each scanner notebook widget.

To override for a specific run without redeploying, pass `notebook_params` directly when triggering the job.

## 1. Catalog & Control Plane

Where the scanners write their findings and where migration plans are stored.

| Setting | Default | Description |
|---|---|---|
| `catalog` | *(required)* | UC catalog that contains your `control_schema`. Must exist before first run. |
| `control_schema` | `uc_hygiene` | Schema for all control tables. Created automatically by `00_control_plane_bootstrap` on every job run. |

**Control tables bootstrapped on first run:**

| Table | Purpose |
|---|---|
| `scan_results` | Daily scan findings — staleness, tag gaps, naming violations. `asset_type` column scopes each finding to a governed asset class (`table`, `model`, `notebook`, `dashboard`, etc.) |
| `certification_state` | Certification lifecycle state per governed asset. `asset_type` discriminator enables certifying models and notebooks alongside tables |
| `migration_plans` | AI-generated migration and remediation plans |
| `bridge_views` | Backward-compat views created during renames |
| `notification_log` | History of all owner notifications sent |
| `cost_attribution_daily` | Daily workload cost attribution for this framework |
| `asset_inventory_snapshot` | Monthly point-in-time table inventory for trend analysis |
| `job_run_history` | Execution history for all UC Steward tasks |
| `exemptions` | Pattern-based scan exemptions (catalog, schema, table, prefix) with expiry |
| `governance_policy` | Runtime policy seeded from `policies/policy.yml` |
| `feature_check_results` | Platform feature enablement check results — status, tier, scope, message per scan date |

**Migration engagement tip**: Use a dedicated catalog (e.g. `data_governance`) so scan results are isolated from the data being scanned. The control schema should be read-accessible to data owners so they can see their own violations.

In [0]:
# ── 1. CATALOG & CONTROL PLANE ─────────────────────────────────────────────
# The catalog + schema where all control tables live.
# These must be consistent across both jobs (daily scan + weekly enrichment).

CATALOG        = ""              # e.g. "data_governance" or "main"
CONTROL_SCHEMA = "uc_hygiene"    # Created on first run if it doesn't exist

# Set in databricks.yml:
#   catalog:        default: ""
#   control_schema: default: "uc_hygiene"
print(f"Control plane: {CATALOG}.{CONTROL_SCHEMA}")

## 2. Discovery Scope

Which catalogs to scan for violations.

| Setting | Default | Description |
|---|---|---|
| `target_catalogs` | *(required)* | Comma-separated list of UC catalog names to scan. |

**Migration engagement tip**: Start with 1-2 catalogs containing the most critical data (e.g. the catalog migrated from HMS). Expand incrementally — scanning everything on day one produces an overwhelming violation count with no prioritization signal.

In [0]:
# ── 2. DISCOVERY SCOPE ──────────────────────────────────────────────────────
# Comma-separated list of catalogs to scan.
# Start narrow — you can always add more catalogs after your first clean run.

TARGET_CATALOGS = ""             # e.g. "prod_catalog" or "prod_catalog,staging_catalog"

# Set in databricks.yml:
#   target_catalogs: default: ""
print(f"Scanning: {TARGET_CATALOGS or '(not set — required)'}")

## 3. Staleness Thresholds

When is a table considered stale? Controls `01_staleness_detector`.

| Setting | Default | Description |
|---|---|---|
| `staleness_days` | `30` | Tables with no read or write activity in this many days are flagged `stale_warning`. Tables at 2× this threshold are flagged `stale_critical`. |

**Migration engagement tip**: 30 days is aggressive for initial discovery — you'll get a lot of noise. Consider 60-90 days for the first pass on a legacy HMS migration to focus on truly dormant objects. Drop to 30 days once the catalog is clean and you're in steady-state governance mode.

**What counts as activity**: Last write (`last_altered`) or, if available, last read from `system.access.table_lineage`. Tables with zero lineage and no writes > `staleness_days` are the safest to deprecate.

In [0]:
# ── 3. STALENESS THRESHOLDS ─────────────────────────────────────────────────
# Days of inactivity before a table is flagged.
# stale_warning  = staleness_days
# stale_critical = staleness_days * 2  (hardcoded in 01_staleness_detector)

STALENESS_DAYS = "30"   # Recommended: 60-90 for initial HMS migration pass
                         #              30 for steady-state governance

# Set in databricks.yml:
#   staleness_days: default: "30"
print(f"Stale warning at: {STALENESS_DAYS} days | Stale critical at: {int(STALENESS_DAYS)*2} days")

## 4. Naming Conventions

Regex patterns used by `03_naming_convention_enforcer`.

| Setting | Default | Description |
|---|---|---|
| `table_name_pattern` | `^[a-z][a-z0-9_]*$` | Tables must match this pattern. Default: lowercase, starts with letter, alphanumeric + underscore. |
| `schema_name_pattern` | `^[a-z][a-z0-9_]*$` | Same rules for schema names. |

The enforcer also checks for **anti-patterns** regardless of the regex: `_copy`, `_backup`, `_old`, `_test`, `_temp`, `_v1/_v2`, date suffixes (`20240101`). These are flagged as `info`-level findings even if the regex passes.

**Migration engagement tip**: The default pattern is strict. If the client has an existing naming standard (e.g. PascalCase, or a prefix like `dim_`/`fact_`), update the patterns before the first scan — otherwise you'll generate hundreds of violations that aren't real governance problems, just historical convention differences.

In [0]:
# ── 4. NAMING CONVENTIONS ────────────────────────────────────────────────────
# Regex patterns for schema and table names.
# Column names use the same pattern (hardcoded in 03_naming_convention_enforcer).

TABLE_NAME_PATTERN  = "^[a-z][a-z0-9_]*$"   # Default: lowercase snake_case
SCHEMA_NAME_PATTERN = "^[a-z][a-z0-9_]*$"   # Same rule for schemas

# Common alternatives:
#   "^[a-z][a-z0-9_]{2,127}$"           — enforce 3-128 char length bounds
#   "^(dim|fact|bridge|staging|raw)_.*" — enforce layer prefix convention
#   "^[a-zA-Z][a-zA-Z0-9_]*$"           — allow mixed case (less strict)

# Set in databricks.yml:
#   table_name_pattern:  default: "^[a-z][a-z0-9_]*$"
#   schema_name_pattern: default: "^[a-z][a-z0-9_]*$"
print(f"Table pattern:  {TABLE_NAME_PATTERN}")
print(f"Schema pattern: {SCHEMA_NAME_PATTERN}")

## 5. Required Tags

Which UC tags every table must have. Controls `02_tag_compliance_scanner`.

| Setting | Default | Description |
|---|---|---|
| `required_table_tags` | `owner,domain,quality_tier` | Comma-separated list of tag keys that must be present on every table. |

**Migration engagement tip**: Three tags cover 90% of governance value for a migration:
- `owner` — who is responsible for this table (email or team name)
- `domain` — which business domain (finance, marketing, operations…)
- `quality_tier` — how reliable is this data (gold, silver, bronze, or certified/uncertified)

Don't add PII sensitivity tags here unless you're ready to act on them — flagging every table without a sensitivity tag in a legacy catalog will produce thousands of findings before you have a process to resolve them.

In [0]:
# ── 5. REQUIRED TAGS ────────────────────────────────────────────────────────
# Every table must have ALL of these tag keys present.
# A table missing even one key is flagged as `missing_required_tag`.

REQUIRED_TABLE_TAGS = "owner,domain,quality_tier"

# Add more only once you have a tagging workflow in place:
#   "owner,domain,quality_tier,sensitivity"       — include PII tier
#   "owner,domain,quality_tier,team,cost_center"  — finance accountability

# Set in databricks.yml:
#   required_table_tags: default: "owner,domain,quality_tier"
print(f"Required tags: {REQUIRED_TABLE_TAGS}")

## 6. Certification & Escalation

Controls how long a violation can sit unacknowledged before escalating. Used by `05_certification_workflow`.

| Setting | Default | Description |
|---|---|---|
| `escalation_days` | `7` | If a table has open violations and no certification action after this many days, it escalates to `overdue` status and triggers a second notification. |

**Migration engagement tip**: 7 days works well for active migration sprints where teams are engaged. For large organizations with slower response cycles, 14-30 days avoids alert fatigue while still maintaining accountability. Set to `3` during a final push before a go-live cutover.

In [0]:
# ── 6. CERTIFICATION & ESCALATION ────────────────────────────────────────────
# Days before an unacknowledged violation escalates to 'overdue'.
# Escalated tables appear at the top of owner notifications.

ESCALATION_DAYS = "7"   # 7 = sprint cadence | 14-30 = enterprise orgs | 3 = go-live crunch

# Set in databricks.yml:
#   escalation_days: default: "7"
print(f"Escalation window: {ESCALATION_DAYS} days")

## 7. Notifications

Where alerts go. Used by `06_owner_notifier` and job-level failure alerts.

| Setting | Default | Description |
|---|---|---|
| `notification_email` | *(required)* | Fallback email for tables with no `owner` tag. Also receives job failure alerts. |
| `slack_webhook_url` | *(empty)* | Optional. If set, summary notifications are also posted to Slack. Leave blank to disable. |

**Migration engagement tip**: Set `notification_email` to a shared team inbox (e.g. `data-governance@company.com`) rather than an individual — violations may sit unresolved for days and personal inboxes fill up fast. Configure the Slack webhook to a `#data-governance` channel so the whole team has visibility without anyone being personally paged.

In [0]:
# ── 7. NOTIFICATIONS ─────────────────────────────────────────────────────────
# notification_email: fallback for unowned tables + job failure alerts.
# slack_webhook_url:  optional channel webhook (leave "" to disable).

NOTIFICATION_EMAIL   = ""   # e.g. "data-governance@company.com"
SLACK_WEBHOOK_URL    = ""   # e.g. "https://hooks.slack.com/services/..."

# Set in databricks.yml:
#   notification_email:  default: ""
#   slack_webhook_url:   default: ""
print(f"Notification email: {NOTIFICATION_EMAIL or '(not set — required)'}")
print(f"Slack webhook:      {'(configured)' if SLACK_WEBHOOK_URL else '(not set — email only)'}")

## 8. AI Settings

Controls AI-generated content in `04_metadata_enricher` and `07_reconciliation_planner`.

| Setting | Default | Description |
|---|---|---|
| `model_name` | `databricks-gemini-3-5-flash` | Foundation model used for rename suggestions and description generation. Any FMAPI-compatible endpoint works. |
| `enable_ai_plans` | `true` | When `true`, reconciliation_planner calls the model to suggest a compliant snake_case rename for each table. When `false`, uses regex heuristic only (faster, no AI cost). |
| `enable_ai_descriptions` | `true` | When `true`, metadata_enricher generates natural language descriptions for tables that have none. |
| `max_plans_per_run` | `20` | Cap on migration plans written per daily run. Prevents runaway costs on first scan of a large catalog. |

**Migration engagement tip**: Run with `enable_ai_plans=false` and `enable_ai_descriptions=false` for the first 2-3 days to get a clean violation baseline without AI cost. Enable AI once you've validated the scan results are accurate and the team is ready to act on rename suggestions.

In [0]:
# ── 8. AI SETTINGS ────────────────────────────────────────────────────────────
# Model used for rename suggestions (07) and description generation (04).
# Any Databricks FMAPI endpoint works — swap to a cheaper model to reduce cost.

MODEL_NAME              = "databricks-gemini-3-5-flash"
ENABLE_AI_PLANS         = "true"   # "false" = regex heuristic only (no AI cost)
ENABLE_AI_DESCRIPTIONS  = "true"   # "false" = skip description generation
MAX_PLANS_PER_RUN       = "20"     # Cap per daily run — raise once baseline is clean

# Available models (test with ai_query to verify access):
#   databricks-gemini-3-5-flash       — fast, low cost, good for structured output
#   databricks-meta-llama-3-3-70b-instruct — open weights, no external data sharing
#   databricks-claude-haiku-4-5       — strong instruction following

# Set in databricks.yml:
#   model_name:             default: "databricks-gemini-3-5-flash"
#   enable_ai_plans:        default: "true"
#   enable_ai_descriptions: default: "true"
#   max_plans_per_run:      default: "20"
print(f"Model: {MODEL_NAME}  |  AI plans: {ENABLE_AI_PLANS}  |  AI descriptions: {ENABLE_AI_DESCRIPTIONS}  |  Max plans/run: {MAX_PLANS_PER_RUN}")

## 9. Integrations (Optional)

### Jira
When set, `07_reconciliation_planner` creates a Jira ticket for each migration plan.

| Setting | Default | Description |
|---|---|---|
| `jira_base_url` | *(empty)* | Your Jira instance URL. e.g. `https://yourorg.atlassian.net` |
| `jira_project_key` | *(empty)* | Jira project where tickets are created. e.g. `DATA` |
| Jira API token | — | Stored as Databricks secret: `dbutils.secrets.get("uc-steward", "jira-api-token")`. Never put in config. |

Leave both empty to disable Jira integration — plans are still written to `migration_plans` in UC.

### Slack
Configure `slack_webhook_url` in section 7. No additional setup needed.

In [0]:
# ── 9. INTEGRATIONS (OPTIONAL) ───────────────────────────────────────────────
# Jira — leave empty to disable. Plans are always written to UC migration_plans.

JIRA_BASE_URL    = ""   # e.g. "https://yourorg.atlassian.net"
JIRA_PROJECT_KEY = ""   # e.g. "DATA"

# Jira API token must be in a secret scope (never hardcode):
#   databricks secrets create-scope uc-steward
#   databricks secrets put-secret uc-steward jira-api-token --string-value <token>

# Set in databricks.yml:
#   jira_base_url:    default: ""
#   jira_project_key: default: ""
print(f"Jira: {'configured' if JIRA_BASE_URL else 'disabled (migration_plans UC table only)'}")

## 10. Feature Enablement Checks

Controls platform feature enforcement in `07_reconciliation_planner`. These checks verify that Databricks best-practice features (Predictive Optimization, Data Classification, Lakehouse Monitoring) are active.

| Setting | Default | Description |
|---|---|---|
| `feature_check_mode` | `standard` | `standard` = skip checks requiring metastore admin; `elevated` = full suite (requires admin grants on system tables) |
| `enable_feature_checks` | `true` | Set `false` to skip feature checks entirely (zero runtime overhead) |

**How it works**: A `CapabilityProbe` runs at startup to detect which system tables the service principal can access. Checks that require inaccessible tables are automatically SKIPPED (not FAILED) — no penalty for missing privileges.

**Score formula**: `PASSED / (PASSED + FAILED) × 100`. SKIPPED checks are excluded from the denominator.

| Check | Tier | What it verifies |
|---|---|---|
| `predictive_optimization_enabled` | standard | Catalog/schema has Predictive Optimization enabled |
| `pii_columns_masked` | standard | All PII-tagged columns have column masks applied |
| `classification_scan_coverage` | elevated | Data Classification has scanned a minimum % of eligible tables |
| `lakehouse_monitor_attached` | elevated | Tables have Lakehouse Monitors attached |

**Migration engagement tip**: Start with `feature_check_mode=standard` (default). Only switch to `elevated` after confirming the SP has metastore admin grants — otherwise elevated checks will all SKIP, adding latency with no value.

In [0]:
# ── 10. FEATURE ENABLEMENT CHECKS ────────────────────────────────────────────
# Mode controls which checks run; enable flag gates the entire subsystem.

FEATURE_CHECK_MODE    = "standard"   # "standard" or "elevated"
ENABLE_FEATURE_CHECKS = "true"       # "false" = skip entirely

# Set in databricks.yml:
#   feature_check_mode:    default: "standard"
#   enable_feature_checks: default: "true"
print(f"Feature checks: {'enabled' if ENABLE_FEATURE_CHECKS == 'true' else 'DISABLED'}  |  Mode: {FEATURE_CHECK_MODE}")

## 10. Identity & Run-As

Which identity the jobs run as. Needed for enterprise deployments where a service principal owns scheduled workloads.

| Setting | Default | Description |
|---|---|---|
| `run_as_user` | *(empty)* | Email of the user or service principal that runs all three jobs. When empty, jobs run as the deploying user. |

**Migration engagement tip**: For production, always set `run_as_user` to a dedicated service principal (e.g. `svc-uc-steward@company.com`). This ensures the job keeps running if the deploying user leaves the org, and makes audit logs cleaner. The SP needs `CAN_MANAGE_RUN` on the jobs and at minimum `SELECT` + `MODIFY` on the control schema.

In [0]:
# ── 10. IDENTITY & RUN-AS ─────────────────────────────────────────────────────
# Which identity runs the jobs. For dev, leave empty (runs as you).
# For prod, use a dedicated service principal.

RUN_AS_USER = ""   # e.g. "svc-uc-steward@company.com" or "" (deploying user)

# Set in databricks.yml:
#   run_as_user: default: ""
print(f"Run-as: {RUN_AS_USER or '(deploying user — set for prod SP)'}")

## 11. Cost Attribution & Framework Tag

Controls how the Phase 5 `08_cost_attribution_tracker` task attributes billing costs. Runs as the final phase of `uc_hygiene_daily_governance`.

| Setting | Default | Description |
|---|---|---|
| `workspace_id` | *(your workspace ID)* | Used to scope `system.billing.usage` queries to this workspace. |
| `cost_lookback_days` | `7` | How many days of billing data to re-aggregate on each run (idempotent delete + reinsert). |
| `cost_currency_code` | `USD` | Currency for price lookup in `system.billing.list_prices`. |
| `cost_min_dbu_threshold` | `0` | Filter out billing records with DBU quantity below this value. Useful to suppress micro-charges from infrastructure noise. |
| `framework_tag` | `uc-steward` | The value of the `framework` custom tag on jobs/clusters that belong to this framework. Used to produce a UC-Hygiene-specific cost breakdown separate from the full workspace rollup. |

**How framework attribution works**: Databricks billing records carry `custom_tags` from the cluster or job that generated the usage. Tagging your jobs with `framework=uc-steward` (done automatically via the `tags:` block in `jobs.yml`) lets the cost tracker filter `system.billing.usage` to show only this framework’s compute costs.

**Migration engagement tip**: Share the framework cost summary with the client’s FinOps team at week 2 — it shows the governance program’s own footprint and helps justify the ongoing investment.

In [0]:
# ── 11. COST ATTRIBUTION & FRAMEWORK TAG ───────────────────────────────────
# workspace_id:          scope billing queries to this workspace
# cost_lookback_days:    days to re-aggregate each run (idempotent)
# framework_tag:         value of the 'framework' custom tag on uc-steward jobs

WORKSPACE_ID               = ""        # auto-populated by jobs; set for manual runs
COST_LOOKBACK_DAYS         = "7"
COST_CURRENCY_CODE         = "USD"
COST_MIN_DBU_THRESHOLD     = "0"
FRAMEWORK_TAG              = "uc-steward"   # must match tags.framework in jobs.yml

# Set in databricks.yml:
#   workspace_id:            default: "7474657986130378"  # your workspace
#   cost_lookback_days:      default: "7"
#   cost_currency_code:      default: "USD"
#   cost_min_dbu_threshold:  default: "0"
#   framework_tag:           default: "uc-steward"
print(f"Cost attribution: {COST_LOOKBACK_DAYS}d lookback | {COST_CURRENCY_CODE} | tag='{FRAMEWORK_TAG}'")

## 12. Data Retention (Monthly Housekeeping)

Controls how long records are kept in the control tables. Enforced by the `uc_hygiene_monthly_housekeeping` job (runs at 04:00 ET on the 1st of each month).

| Setting | Default | Description |
|---|---|---|
| `scan_results_retention_days` | `90` | `scan_results` rows older than this are deleted. Only resolved findings are pruned aggressively. |
| `plan_retention_days` | `180` | `migration_plans` in `completed` or `cancelled` status older than this are deleted. Open plans are never pruned. |
| `cost_attribution_retention_days` | `365` | `cost_attribution_daily` rows older than this are deleted. Keep at least 365 days for year-over-year comparison. |

**What else the monthly job does**:
- Sets `certification_state` to `overdue` for tables whose `next_review_date` has passed
- Captures a point-in-time `asset_inventory_snapshot` for trend analysis (tag coverage %, table count over time)
- Emits a governance health report: open violations by severity, cert state distribution, plan backlog, notification response rate
- Prunes acknowledged `notification_log` records older than `plan_retention_days`

**Migration engagement tip**: Keep `scan_results_retention_days` at 90 for the first 6 months — you’ll want to look back at the violation trend to show progress. After go-live, 30 days is sufficient.

In [0]:
# ── 12. DATA RETENTION (MONTHLY HOUSEKEEPING) ───────────────────────────────
# How long to keep control table records. Enforced by Monthly Housekeeping job.

SCAN_RESULTS_RETENTION_DAYS     = "90"    # 90d for migration, 30d steady-state
PLAN_RETENTION_DAYS             = "180"   # keep completed plans 6 months
COST_ATTRIBUTION_RETENTION_DAYS = "365"   # 1yr for YoY cost comparison

# Set in databricks.yml:
#   scan_results_retention_days:     default: "90"
#   plan_retention_days:             default: "180"
#   cost_attribution_retention_days: default: "365"
#   housekeeping_monthly_cron:       default: "0 0 4 1 * ?"  # 04:00 ET, 1st of month
print(f"Retention: scan={SCAN_RESULTS_RETENTION_DAYS}d | plans={PLAN_RETENTION_DAYS}d | cost={COST_ATTRIBUTION_RETENTION_DAYS}d")

## 13. Recommended First-Week Settings

Copy these into `databricks.yml` for a typical HMS → UC migration engagement. Tuned to minimize noise on first contact with a messy legacy catalog.

| Setting | First-week value | Why |
|---|---|---|
| `staleness_days` | `90` | 30 days flags too much on a legacy catalog — focus on truly dormant objects first |
| `enable_ai_plans` | `false` | Get a clean violation baseline before spending AI tokens |
| `enable_ai_descriptions` | `false` | Same — validate scans are accurate first |
| `max_plans_per_run` | `20` | Hard cap while you're still calibrating; raise to 50-100 once baseline is clean |
| `escalation_days` | `14` | Give teams breathing room on first contact; tighten to 7 once the process is established |
| `required_table_tags` | `owner,domain` | Start with 2 tags — adding `quality_tier` day one creates hundreds of findings before anyone has a tagging workflow |
| `table_name_pattern` | *(confirm with client)* | Ask what naming convention they actually want before scanning — otherwise you flag legitimate historical names |
| `run_as_user` | SP email | Set to a dedicated SP for prod — never use a personal account for scheduled jobs |
| `framework_tag` | `uc-steward` | Leave as default; matches the `tags.framework` value in `jobs.yml` |
| `scan_results_retention_days` | `90` | Keep at 90 for the engagement — you want the violation trend history |

**Day 3 flip**: Once you've reviewed the first scan results with the client and validated the violation counts make sense, flip `enable_ai_plans=true` and `enable_ai_descriptions=true`, drop `staleness_days` to 60, and redeploy.

**Day 7 flip**: Add `quality_tier` to `required_table_tags`, drop `escalation_days` to 7, raise `max_plans_per_run` to 50.

**Month 1 housekeeping**: Let the monthly job run once before tuning retention days. Review the health report it emits — it shows tag coverage trend and notification response rate that are useful for the first executive check-in.

In [0]:
# ── 13. RECOMMENDED FIRST-WEEK SETTINGS ─────────────────────────────────────
# Copy this block into databricks.yml variables section for a clean first run.
# Designed for HMS → UC migration engagements with messy legacy catalogs.
#
# variables:
#   catalog:                default: "<your_catalog>"       # REQUIRED
#   control_schema:         default: "uc_hygiene"
#   target_catalogs:        default: "<migrated_catalog>"   # REQUIRED — start with 1
#   notification_email:     default: "<team_inbox@co.com>"  # REQUIRED
#
#   staleness_days:         default: "90"     # loosen for initial pass
#   escalation_days:        default: "14"     # give teams breathing room
#   required_table_tags:    default: "owner,domain"  # 2 tags to start
#
#   enable_ai_plans:        default: "false"  # flip to true on day 3
#   enable_ai_descriptions: default: "false"  # flip to true on day 3
#   max_plans_per_run:      default: "20"     # raise to 50-100 once clean
#
#   # Identity (set to SP for prod)
#   run_as_user:            default: "svc-uc-steward@company.com"
#
#   # Cost attribution (defaults are fine for week 1)
#   framework_tag:          default: "uc-steward"
#   cost_lookback_days:     default: "7"
#   workspace_id:           default: "<your_workspace_id>"
#
#   # Retention (defaults are fine for week 1)
#   scan_results_retention_days:     default: "90"
#   plan_retention_days:             default: "180"
#   cost_attribution_retention_days: default: "365"
#
#   # Leave these as defaults for week 1:
#   table_name_pattern:  default: "^[a-z][a-z0-9_]*$"  # CONFIRM with client first
#   schema_name_pattern: default: "^[a-z][a-z0-9_]*$"
#   slack_webhook_url:   default: ""
#   jira_base_url:       default: ""
#   jira_project_key:    default: ""

# ── WEEK 1 CHECKLIST ─────────────────────────────────────────────────────────
checklist = [
    ("catalog + control_schema",       "Fill in databricks.yml"),
    ("target_catalogs",                "Start with 1 catalog (the migrated one)"),
    ("notification_email",             "Use a team inbox, not an individual"),
    ("run_as_user",                    "Set to SP for prod; empty = deploying user for dev"),
    ("workspace_id",                   "Set to your workspace ID for cost attribution"),
    ("naming patterns",                "Confirm convention with client before first scan"),
    ("bundle deploy",                  "databricks bundle deploy --target prod"),
    ("manual trigger",                 "Trigger uc_hygiene_daily_governance, review scan_results"),
    ("enable daily schedule",          "Only after validating violation counts make sense"),
    ("enable monthly housekeeping",    "Enable uc_hygiene_monthly_housekeeping schedule"),
    ("day 3: flip AI on",              "enable_ai_plans=true, enable_ai_descriptions=true, redeploy"),
    ("day 7: tighten governance",      "Add quality_tier tag, drop escalation_days to 7, max_plans to 50"),
    ("month 1: review health report",  "Monthly housekeeping emits tag coverage trend + notification response rate"),
]

print("Week 1 setup checklist:")
for step, note in checklist:
    print(f"  [ ] {step:<35} — {note}")

## 14. Deployment

### First-time setup
1. Clone this repo and `cd` into the project directory
2. Fill in all `(required)` settings in `databricks.yml` → `variables` section
3. Run `databricks bundle deploy --target prod`
4. Enable the job schedule in the Lakeflow Jobs UI (schedules are paused by default)
5. Trigger a manual run of `uc_hygiene_daily_governance` to verify control tables are created

### Changing a setting
- **Persistent change**: Edit `databricks.yml` → re-run `databricks bundle deploy --target prod`
- **One-off override**: Pass `notebook_params` when triggering the job via API or UI

### Targets
| Target | Purpose |
|---|---|
| `dev` | Developer sandbox — job names prefixed `[dev <username>]`, uses `uc_hygiene_dev` schema |
| `prod` | Production — job names unprefixed, uses `uc_hygiene` schema |

### Jobs deployed
| Job | Schedule | Purpose |
|---|---|---|
| `uc_hygiene_daily_governance` | 06:00 ET daily | Scan → remediate → notify → cost attribution |
| `uc_hygiene_weekly_enrichment` | 03:00 ET Sunday | AI descriptions + PII auto-tagging |
| `uc_hygiene_monthly_housekeeping` | 04:00 ET 1st of month | Pruning, cert expiry, health report |

### Control table locations
All findings are queryable in UC:
```sql
-- Open violations
SELECT * FROM <catalog>.uc_hygiene.scan_results
WHERE resolved_at IS NULL ORDER BY scan_date DESC LIMIT 100;

-- Pending migration plans (highest priority first)
SELECT * FROM <catalog>.uc_hygiene.migration_plans
WHERE plan_status = 'pending' ORDER BY priority_score DESC;

-- Overdue certifications
SELECT * FROM <catalog>.uc_hygiene.certification_state
WHERE current_status = 'overdue';

-- This framework's daily cost (last 30 days)
SELECT usage_date, attribution_label,
       ROUND(SUM(usage_quantity),2) AS dbus,
       ROUND(SUM(estimated_cost),4) AS est_usd
FROM <catalog>.uc_hygiene.cost_attribution_daily
WHERE usage_date >= current_date() - INTERVAL 30 DAYS
  AND framework_tag = 'uc-steward'
GROUP BY 1,2 ORDER BY 1 DESC, 4 DESC;

-- Tag coverage trend (from monthly snapshots)
SELECT snapshot_date,
       COUNT(*) AS tables,
       ROUND(100.0*SUM(CASE WHEN has_owner_tag   THEN 1 END)/COUNT(*),1) AS owner_pct,
       ROUND(100.0*SUM(CASE WHEN has_domain_tag  THEN 1 END)/COUNT(*),1) AS domain_pct,
       ROUND(100.0*SUM(CASE WHEN has_quality_tag THEN 1 END)/COUNT(*),1) AS quality_pct
FROM <catalog>.uc_hygiene.asset_inventory_snapshot
GROUP BY 1 ORDER BY 1 DESC;
```